In [7]:
import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, average_precision_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

In [8]:
# Cell 2: Load datasets and define key columns
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

outcome_col = "visit"
treatment_col = "treatment"

exclude_from_features = {outcome_col, "conversion", "exposure"}
feature_cols = [c for c in train_df.columns if c not in exclude_from_features]
t_learner_feature_cols = [c for c in feature_cols if c != treatment_col]

X_train = train_df[t_learner_feature_cols].copy()
y_train = train_df[outcome_col].astype(int)
w_train = train_df[treatment_col].astype(int)
X_test = test_df[t_learner_feature_cols].copy()
y_test = test_df[outcome_col].astype(int)
w_test = test_df[treatment_col].astype(int)

print("Train rows:", len(train_df), "| Test rows:", len(test_df))

Train rows: 582540 | Test rows: 200000


In [9]:
# Cell 3: Train LightGBM T-learner with 50-trial Optuna search (optimize AUPRC)
numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

# Use pandas categorical dtype so LightGBM can handle categorical features natively.
for col in categorical_cols:
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype("string")
    categories = pd.Index(combined.dropna().unique())
    X_train[col] = pd.Categorical(X_train[col].astype("string"), categories=categories)
    X_test[col] = pd.Categorical(X_test[col].astype("string"), categories=categories)

X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(
    X_train,
    y_train,
    w_train,
    test_size=0.2,
    random_state=42,
    stratify=(y_train.astype(str) + "_" + w_train.astype(str)),
)

def objective(trial):
    params = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    treated_model = LGBMClassifier(**params)
    control_model = LGBMClassifier(**params)

    treated_mask = w_tr == 1
    control_mask = w_tr == 0

    treated_model.fit(X_tr.loc[treated_mask], y_tr.loc[treated_mask], categorical_feature=categorical_cols)
    control_model.fit(X_tr.loc[control_mask], y_tr.loc[control_mask], categorical_feature=categorical_cols)

    mu1_val = treated_model.predict_proba(X_val)[:, 1]
    mu0_val = control_model.predict_proba(X_val)[:, 1]
    factual_val_proba = np.where(w_val.values == 1, mu1_val, mu0_val)
    return average_precision_score(y_val, factual_val_proba)

study = optuna.create_study(direction="maximize", study_name="t_learner_visit_auprc")
study.optimize(objective, n_trials=50, show_progress_bar=False)

best_params = study.best_params.copy()
best_params.update({"objective": "binary", "random_state": 42, "n_jobs": -1, "verbosity": -1})

t_learner_treated = LGBMClassifier(**best_params)
t_learner_control = LGBMClassifier(**best_params)

treated_mask_full = w_train == 1
control_mask_full = w_train == 0

t_learner_treated.fit(X_train.loc[treated_mask_full], y_train.loc[treated_mask_full], categorical_feature=categorical_cols)
t_learner_control.fit(X_train.loc[control_mask_full], y_train.loc[control_mask_full], categorical_feature=categorical_cols)

print(f"Best validation AUPRC: {study.best_value:.4f}")
print("LightGBM T-learner trained using best Optuna trial on full training data.")

[I 2026-04-07 17:15:44,600] A new study created in memory with name: t_learner_visit_auprc
[I 2026-04-07 17:16:05,361] Trial 0 finished with value: 0.5119773112004479 and parameters: {'n_estimators': 195, 'learning_rate': 0.02042822262595695, 'num_leaves': 84, 'max_depth': 5, 'min_child_samples': 26, 'subsample': 0.7127296688198465, 'colsample_bytree': 0.7182634582202784, 'reg_alpha': 0.0013345765558918693, 'reg_lambda': 1.499912415597798e-05}. Best is trial 0 with value: 0.5119773112004479.
[I 2026-04-07 17:16:13,170] Trial 1 finished with value: 0.5050299355042072 and parameters: {'n_estimators': 217, 'learning_rate': 0.15409421726315944, 'num_leaves': 20, 'max_depth': 4, 'min_child_samples': 108, 'subsample': 0.6214381413860524, 'colsample_bytree': 0.7981192086123698, 'reg_alpha': 0.09412161508362532, 'reg_lambda': 0.010685251820669482}. Best is trial 0 with value: 0.5119773112004479.
[I 2026-04-07 17:16:18,652] Trial 2 finished with value: 0.5102745669459164 and parameters: {'n_est

Best validation AUPRC: 0.5132
LightGBM T-learner trained using best Optuna trial on full training data.


In [10]:
# Cell 4: Evaluate factual performance on test.csv
mu1_test = t_learner_treated.predict_proba(X_test)[:, 1]
mu0_test = t_learner_control.predict_proba(X_test)[:, 1]
factual_proba = np.where(w_test.values == 1, mu1_test, mu0_test)
factual_pred = (factual_proba >= 0.5).astype(int)

precision = precision_score(y_test, factual_pred, zero_division=0)
recall = recall_score(y_test, factual_pred, zero_division=0)
accuracy = accuracy_score(y_test, factual_pred)
auprc = average_precision_score(y_test, factual_proba)

print("Test metrics on test.csv")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"AUPRC:     {auprc:.4f}")

Test metrics on test.csv
Precision: 0.6212
Recall:    0.3139
Accuracy:  0.9592
AUPRC:     0.5110


In [11]:
# Cell 5: Counterfactual conversion predictions from T-learner
mu1 = t_learner_treated.predict_proba(X_test)[:, 1]  # P(conversion | do(treatment=1), x)
mu0 = t_learner_control.predict_proba(X_test)[:, 1]  # P(conversion | do(treatment=0), x)
uplift = mu1 - mu0

results = test_df.copy()
results["p_conversion_treated"] = mu1
results["p_conversion_control"] = mu0
results["uplift"] = uplift
results

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure,p_conversion_treated,p_conversion_control,uplift
0,23.032518,10.059654,8.214383,4.679882,10.280525,4.115453,-12.422866,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0,0.002572,0.000698,0.001874
1,25.600311,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0,0.001106,0.000438,0.000668
2,19.523195,10.059654,8.639891,3.359763,10.280525,4.115453,-7.301017,4.833815,3.878372,25.240993,5.300375,-0.168679,1,0,0,0,0.089041,0.060166,0.028875
3,12.616365,10.059654,8.422836,4.679882,10.280525,4.115453,0.294443,4.833815,3.835851,18.380112,5.300375,-0.168679,1,0,0,0,0.166139,0.184541,-0.018402
4,20.454730,10.059654,8.847605,3.907662,10.280525,4.115453,-10.006574,4.833815,3.899112,13.190056,5.300375,-0.168679,1,0,0,0,0.038336,0.020708,0.017628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,12.616365,10.059654,9.040947,4.679882,10.280525,4.115453,0.294443,4.833815,3.943716,13.190056,5.300375,-0.168679,1,0,0,0,0.003616,0.003580,0.000036
199996,24.811454,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,0,0,0,0,0.001030,0.000402,0.000627
199997,22.928756,10.059654,8.378748,4.679882,10.280525,3.013064,-15.877431,11.560131,3.771810,46.672202,5.300375,-0.168679,0,0,0,0,0.403939,0.387960,0.015979
199998,12.616365,10.059654,8.841462,4.679882,10.280525,4.115453,0.294443,4.833815,3.943716,13.190056,5.300375,-0.168679,1,0,0,0,0.005425,0.004314,0.001111


In [12]:
# Cell 6: Save best T-learner models to PKL
import pickle

t_learner_bundle = {
    "treated_model": t_learner_treated,
    "control_model": t_learner_control,
    "feature_cols": t_learner_feature_cols,
    "categorical_cols": categorical_cols,
    "treatment_col": treatment_col,
    "outcome_col": outcome_col,
    "best_params": best_params,
    "best_validation_auprc": study.best_value,
    "n_optuna_trials": 50,
}

with open("models/t_learner-visit.pkl", "wb") as f:
    pickle.dump(t_learner_bundle, f)

print("Saved best Optuna-tuned model: t_learner-visit.pkl")

Saved best Optuna-tuned model: t_learner-visit.pkl
